# 1. Model Training, Benchmark, and Algorithmic Comparison

## 2. Objective
Train and benchmark multiple machine learning regression models on chronologically split training and test data (80/20). Compare performance using MAE, MSE, RMSE, and $R^2$.

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt

processed_path = "../datasets/processed/traffic_processed.csv"
df = pd.read_csv(processed_path)
df['date_time'] = pd.to_datetime(df['date_time'])
df = df.sort_values('date_time').reset_index(drop=True)

# 80/20 Chronological Split
split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()

features = ['hour', 'day_of_week', 'month', 'is_weekend', 'is_rush_hour', 'temp', 'rain_1h', 'snow_1h', 'clouds_all', 'weather_main']
target = 'traffic_volume'

X_train, y_train = train_df[features], train_df[target]
X_test, y_test = test_df[features], test_df[target]

print(f"Training observations: {len(X_train):,} ({train_df['date_time'].min().strftime('%Y-%m')} to {train_df['date_time'].max().strftime('%Y-%m')})")
print(f"Testing observations:  {len(X_test):,} ({test_df['date_time'].min().strftime('%Y-%m')} to {test_df['date_time'].max().strftime('%Y-%m')})")

## 3. Preprocessing Pipeline Definition

In [ ]:
numeric_features = ['hour', 'day_of_week', 'month', 'temp', 'rain_1h', 'snow_1h', 'clouds_all']
categorical_features = ['weather_main']
binary_features = ['is_weekend', 'is_rush_hour']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
        ('passthrough', 'passthrough', binary_features)
    ]
)
print("Preprocessing pipeline configured.")

## 4. Benchmark Models Execution

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42, max_depth=12),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=150, learning_rate=0.1, max_depth=6, random_state=42)
}

results = []

for name, model in models.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    
    results.append({
        'Model': name,
        'MAE': round(mae, 2),
        'MSE': round(mse, 2),
        'RMSE': round(rmse, 2),
        'R2': round(r2, 4)
    })
    print(f"✓ Trained and evaluated {name}: R² = {r2:.4f}, RMSE = {rmse:.2f}")

results_df = pd.DataFrame(results)

## 5. Model Comparison Summary

In [ ]:
display(results_df.sort_values(by='R2', ascending=False).reset_index(drop=True))

plt.figure(figsize=(10, 4))
sns.barplot(data=results_df, x='Model', y='R2', palette='Blues_r', edgecolor='black')
plt.title("Model Comparison on Chronological Test Split ($R^2$ Score)", fontsize=13, fontweight='bold')
plt.ylabel("R² Score (Variance Explained)")
plt.ylim(0, 1.0)
for idx, val in enumerate(results_df['R2']):
    plt.text(idx, val + 0.02, f"{val:.4f}", ha='center', fontweight='bold')
plt.show()